# EDA 5 - Yee & Dennett (2001-2011) vs Flow Cascade Typology with 2011 OD data

In [ ]:
# ── imports ──────────────────────────────────────────
import sys
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})

In [ ]:
# ── paths ──────────────────────────────────────────
ROOT       = here()
sys.path.insert(0, str(ROOT))
DATA_DIR   = ROOT / 'data'
OUT_DIR    = ROOT / 'outputs'
GEO_PATH   = DATA_DIR / 'london_msoa_2011.geojson'
FIG_DIR    = OUT_DIR / 'comparison_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

YEE_LABELS = DATA_DIR / 'yee_LSOA_labels_forMapping.csv'
LOOKUP     = OUT_DIR / 'lsoa11_to_msoa11.csv'
EDA4       = OUT_DIR  / 'eda4_results_for_phase3_20260626.csv'

In [ ]:
# ── theme ──────────────────────────────────────────
from map_utils import load_london_msoa, plot_london_categorical

# FLOW_TYP = 'Typ_A_11'                       # 2011 leg, London-only frame (matches Yee)
FLOW_TYP_A = 'Typ_A_11'
FLOW_TYP_C = 'Typ_C_11'

GENTRIFYING = {'GEN'}                        # widen to {'GEN','IUP'} for a broader definition


In [ ]:
# ── Comparison helpers ────────────────────────────────────────────────────────

## measuring association or agreement between 2 categorical classifications
def cramers_v(x, y):
    """Bias-corrected Cramér's V (Bergsma 2013)."""
    tab = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tab, correction=False)[0]
    n = tab.values.sum(); r, k = tab.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    denom = min(kc - 1, rc - 1)
    return np.sqrt(phi2 / denom) if denom > 0 else np.nan


def modal_aggregate(lsoa_df, class_col):
    g = lsoa_df.groupby('msoa11cd')
    modal = g[class_col].agg(lambda s: s.value_counts().index[0])
    conf = g[class_col].agg(lambda s: s.value_counts().iloc[0] / len(s))
    return modal.rename('yee_modal'), conf.rename('modal_confidence')

In [ ]:
# ── load + aggregate Yee LSOA labels to MSOA ───────────────────────
## flow Frame A comparison - London-only flows + London-only ladder
e4 = pd.read_csv(EDA4)
frame = set(e4['msoa11cd'])

yee = pd.read_csv(YEE_LABELS).rename(columns={'LSOA_Code': 'lsoa11cd'})
lk = pd.read_csv(LOOKUP)[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
yg = yee.merge(lk, on='lsoa11cd', how='inner')
yg = yg[yg['msoa11cd'].isin(frame)].copy()
print(f'Yee LSOAs in frame: {len(yg)} across {yg["msoa11cd"].nunique()}/{len(frame)} MSOAs')

modal, conf = modal_aggregate(yg, 'Class_2_status')
has_gen = (yg[yg['Class_2_status'].isin(GENTRIFYING)]
           .groupby('msoa11cd').size().reindex(modal.index, fill_value=0) > 0).rename('has_gen')

df = e4.merge(pd.concat([modal, conf, has_gen], axis=1), left_on='msoa11cd',
              right_index=True, how='left')
sub_a = df.dropna(subset=['yee_modal', FLOW_TYP_A]).copy()
print(f'MSOAs with a Yee label: {sub_a.shape[0]}/{len(df)}')


### Explanation:

- `has_gen` is a binary, containing at least 1 GEN LSO.
    - ~207 MSOAs qualify.
    - In the map below to see the divergence, we used the `yee_modal` to mark "GEN" labels.

- `yee_modal` is the plurality class, where an MSOA is "GEN" only if GEN is the most common label among it's LSOAs
    - only 41 MSOAs are "GEN", more strict.
    - Yee's labels are at LAOS level, each MSOA holds ~5 LSOAs.


We will use `yee_modal` with strict aggregation for the later analysis, but loosening to any-GEN-LSOA weakens the lift to ~1.3× but **preserves the direction**.

In [ ]:
# ── load + aggregate Yee LSOA labels to MSOA ───────────────────────
## flow Frame C comparison - London-external flows + national-frame ladder
e4 = pd.read_csv(EDA4)
frame = set(e4['msoa11cd'])

yee = pd.read_csv(YEE_LABELS).rename(columns={'LSOA_Code': 'lsoa11cd'})
lk = pd.read_csv(LOOKUP)[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
yg = yee.merge(lk, on='lsoa11cd', how='inner')
yg = yg[yg['msoa11cd'].isin(frame)].copy()
print(f'Yee LSOAs in frame: {len(yg)} across {yg["msoa11cd"].nunique()}/{len(frame)} MSOAs')

modal, conf = modal_aggregate(yg, 'Class_2_status')
has_gen = (yg[yg['Class_2_status'].isin(GENTRIFYING)]
           .groupby('msoa11cd').size().reindex(modal.index, fill_value=0) > 0).rename('has_gen')

df = e4.merge(pd.concat([modal, conf, has_gen], axis=1), left_on='msoa11cd',
              right_index=True, how='left')
sub_c = df.dropna(subset=['yee_modal', FLOW_TYP_C]).copy()
print(f'MSOAs with a Yee label: {sub_c.shape[0]}/{len(df)}')

In [ ]:
# ── association between the two categorical schemes ────────────────
## flow Frame A comparison - London-only flows + London-only ladder
V = cramers_v(sub_a[FLOW_TYP_A], sub_a['yee_modal'])
ari = adjusted_rand_score(sub_a[FLOW_TYP_A].astype('category').cat.codes,
                          sub_a['yee_modal'].astype('category').cat.codes)
print(f"Cramér's V (bias-corrected) = {V:.3f}   |   Adjusted Rand Index = {ari:.3f}")
print('NOTE: Yee is ~71% stable (STB); single-number stats are deflated by that')

print('Cross-tab (row %):')
print((pd.crosstab(sub_a[FLOW_TYP_A], sub_a['yee_modal'], normalize='index') * 100).round(1).to_string())


### Interpretation (frame A)

- Under **London-only ladder**, **Majority (78%-92%) of MSOAs are classified as "Stable" in Yee's work, no matter which typology.**
- Among **gentrification** labels, majority MSOAs were "Cascade-led" (8.5%), while there is only a tiny subsets of MSOAs were "Counter-led" (0.9%).
> ***Where are those 0.9% counter-led but gen?***

In [ ]:
# ── association between the two categorical schemes ────────────────
## flow Frame A comparison - London-only flows + London-only ladder
V = cramers_v(sub_c[FLOW_TYP_C], sub_c['yee_modal'])
ari = adjusted_rand_score(sub_c[FLOW_TYP_C].astype('category').cat.codes,
                          sub_c['yee_modal'].astype('category').cat.codes)
print(f"Cramér's V (bias-corrected) = {V:.3f}   |   Adjusted Rand Index = {ari:.3f}")
print('NOTE: Yee is ~71% stable (STB); single-number stats are deflated by that')

print('Cross-tab (row %):')
print((pd.crosstab(sub_c[FLOW_TYP_C], sub_c['yee_modal'], normalize='index') * 100).round(1).to_string())


### Interpretation (frame C)

- ***2 statistical results suggest Yee's labels and my typology don't influence, mirror, or align with each other in any meaningful way.***
    - Cramer's V is only 0.103, indicating there is a very weak mathematical association between 2.
    - Like Moran's I, Adjusted Rand Index is nearly zero, meaning they agree with each other no better than just assigned the labels blindly at random.
- Coverting to national ladder and with external flows, **the pattern of majority of all 4 flow typology classified as "Stable" are remained.** The percentages even increased a bit, compared with frame A.
- Among gentrification labels, "Cascade-led" MSOAs decreased a bit (8.5 -> 7.5).

In [ ]:
# ── link Yee class to IMD validation + convergence check ───────────
## Frame C
print('Mean IMD_Pctile_Change by Yee modal class:')
print(sub_c.groupby('yee_modal')['IMD_Pctile_Change']
      .agg(['mean', 'median', 'count']).round(4).to_string())

print('\nConvergence: IMD change by (flow cascade vs. Yee-gentrifying):')
sub_c['flow_cascade'] = sub_c[FLOW_TYP_C] == 'Cascade-led'
print(sub_c.groupby(['flow_cascade', 'has_gen'])['IMD_Pctile_Change']
      .agg(['mean', 'median', 'count']).round(4).to_string())

### Interpretation of Convergence table:

"Does cascade flow add anythong on top of Yee?"

- F-F is the baseline.
- F-T is Yee alone, with +0.056 jump from baseline.
- T-F is cascade alone, with +0.02 from baseline. 
    - The median is negative --> interaction is negative.
- T-T is Yee + cascade, with +0.0169.

GEN-without-cascade (0.043) improves more than GEN-with-cascade (0.017)
- ***Cascade does NOT reinforce gentrification.*** 
- ***Cascade flow is NOT the engine of IMD-measured gentrification.***

In [ ]:
sub = df.dropna(subset=['yee_modal', 'Typ_C_11'])

# LIFT: how concentrated is each flow type within each Yee class vs its base rate
col  = pd.crosstab(sub['Typ_C_11'], sub['yee_modal'], normalize='columns')
base = sub['Typ_C_11'].value_counts(normalize=True)
print("Column % P(flow | Yee):\n", (col * 100).round(1).to_string())
print("\nLIFT (>1 = over-represented):\n", col.div(base, axis=0).round(2).to_string())

# VALIDATION side by side
print("\nIMD change by Yee class:\n",
      sub.groupby('yee_modal')['IMD_Pctile_Change'].agg(['mean','median','count']).round(4).to_string())
print("\nIMD change by flow typology:\n",
      sub.groupby('Typ_C_11')['IMD_Pctile_Change'].agg(['mean','median','count']).round(4).to_string())

In [ ]:
def kw_effect(frame, group_col, value_col='IMD_Pctile_Change'):
    s = frame.dropna(subset=[group_col, value_col])
    grps = [g[value_col].values for _, g in s.groupby(group_col) if len(g) > 1]
    H, p = stats.kruskal(*grps)
    k, n = len(grps), len(s)
    eps2 = (H - k + 1) / (n - k)          # epsilon-squared (0–1, share of rank variance)
    return H, p, eps2, n, k

print("Does IMD change differ across classes?  (same rows, same test)\n")
for label, col in [('Yee modal class', 'yee_modal'), ('Flow typology (C, 2011)', 'Typ_C_11')]:
    H, p, eps2, n, k = kw_effect(sub_c, col)
    print(f"{label:<26s}  H={H:5.1f}  p={p:.1e}  epsilon^2={eps2:.3f}  (n={n}, k={k})")

# direction check: mean IMD change ordered, so you can read the sign pattern
print("\nMean IMD_Pctile_Change by class (for direction):")
for col in ['yee_modal', 'Typ_C_11']:
    print(f"\n  {col}:")
    print(sub_c.groupby(col)['IMD_Pctile_Change'].mean().round(4).sort_values(ascending=False).to_string())

### Interpretaion of table:

***Yee's gentrifying areas, where they carry a flow signature, lean cascade and essentially never counter.***
- only strong, clean signal is the "GEN" column.
    - modal-GEN MSOAs are ~1.8 times over-represented in cascade (lift = 1.8).
    - modal-GEN MSOAs are ~11 times under-represented in counter (lift = 0.09).

***Yee-GEN tracks IMD improvement, directionally as a gentrification signal,  while flow typology does NOT.***
- **We can't treat cascade flow as a proxy for IMD-defined gentrification.**

***Both attribute and flow typologies are weak predictors of IMD change*** (epsilon^2 are both ~0.04).
    - Yee-GEN is not strong predicts!
- Yee's labels are monotonic, and predicts the direction of deprivation change even if weakly.
- Flow typology is NOT a gentrification gradient

***Flow method's IMD signal lived in the counter direction, not the cascade direction.***


---

In [ ]:
# ── divergence categories (flow vs Yee attribute) ──────────────────
def yee_divergence(row):
    # g = bool(row['has_gen'])
    g = (row['yee_modal'] == 'GEN'); c = row[FLOW_TYP_C]
    if g and c == 'Cascade-led':       return 'Agree: GEN & cascade'
    if g and c != 'Cascade-led':       return 'Gap: Yee-GEN, non-cascade flow'   # attribute-only
    if (not g) and c == 'Cascade-led': return 'Flow-only: cascade, no Yee-GEN'
    return 'Neither'

df['YeeDiv'] = df.apply(yee_divergence, axis=1)
print(df['YeeDiv'].value_counts().to_string())

YEE_DIV_COLORS = {
    'Gap: Yee-GEN, non-cascade flow': '#d73027',   # red  — attribute flags, flow doesn't
    'Flow-only: cascade, no Yee-GEN': '#4575b4',    # blue — flow flags, attribute doesn't
    'Agree: GEN & cascade':          '#1a9850',     # green — convergence
    'Neither':                       '#e0e0e0',     # grey
}


In [ ]:
# ── MAP 1 · Yee–flow divergence (the research gap) ─────────────────
gdf = load_london_msoa(GEO_PATH, df)
fig, ax = plt.subplots(figsize=(11, 9))
plot_london_categorical(
    gdf, column='YeeDiv',
    title='Where flow and Yee (attribute) methods disagree (2011)',
    color_dict=YEE_DIV_COLORS, ax=ax)
if ax.get_legend():
    ax.get_legend().remove()
fig.legend(handles=[mpatches.Patch(color=c, label=l) for l, c in YEE_DIV_COLORS.items()],
           loc='lower center', bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig01_map_yee_divergence_2011.png', bbox_inches='tight')
plt.show()


### Interpretation

The attribute-flow convergence detected is **specific to 2011 and to inner London**.
- ***Yee's method and my method agrees where and when classic inner-Londonn gentrification was happening.***
- The divergence attributes to the flow regime restructured around it.

In the next section below, London shifted to a more clear counter-dominated, outer-shifted regime.

---

## Check alignment 

If inner classic gentrification areas between Yee's labels and my typology detection are the same.

2 areas persist over periods as cascade dominated in EDA 4 Part 4. -- Camden and Tower Hamlets.

In [ ]:
PERSIST    = {'E02000191': 'Camden 026', 'E02000873': 'Tower Hamlets 010'}
 
CLASS2 = {'GEN':'Gentrifying','IUP':'Incumbent upgrading','NRW':'New-build re-urbanisation',
          'DEC':'Declining','STB':'Stable'}
CLASS3 = {'SupGen':'Super-gentrification','MargGen':'Marginal gentrification',
          'MainGen':'Mainstream gentrification','IUP':'Incumbent upgrading',
          'NRW':'New-build re-urbanisation','DEC':'Declining','STB':'Stable'}
 
for code, name in PERSIST.items():
    sub = yg[yg['msoa11cd'] == code].copy()
    sub['Class_2'] = sub['Class_2_status'].map(CLASS2).fillna(sub['Class_2_status'])
    sub['Class_3'] = sub['Class_3_status'].map(CLASS3).fillna(sub['Class_3_status'])
    modal   = sub['Class_2_status'].value_counts().idxmax()
    has_gen = bool((sub['Class_2_status'] == 'GEN').any())
    print('=' * 70)
    print(f'{name}   ({code})   —   {len(sub)} constituent LSOAs')
    print('=' * 70)
    print(sub[['lsoa11cd','Class_1_status','Class_2','Class_3']].to_string(index=False))
    print(f'\n  Modal Class-2 label   : {modal} ({CLASS2.get(modal, modal)})')
    print(f'  Contains any GEN LSOA?: {has_gen}')
    print(f'  Class-2 composition   : {sub["Class_2_status"].value_counts().to_dict()}\n')
 
 

### Interpretation

**Camden 026**
- merged 4 LSOAs
    - 1 marginal gentrification (weakest gen subtype)
    - 3 stable

**Tower Hamlets 010**
- merged 5 LSOAs
    - 4 stable
    - 1 new-build re-urbanisation
    - 0 gen labels 

Neither of these 2 areas is a clean Yee-GEN match.
- ***my flow method flags persistent cascades that Yee's attribute method does not label GEN***.

In [ ]:
# ── MAP 2 · Yee modal class (attribute typology for reference) ─────
YEE_CLASS_COLORS = {'GEN': '#762a83', 'IUP': '#af8dc3', 'NRW': '#7fbf7b',
                    'DEC': '#1b7837', 'STB': '#e7e7e7'}

gdf = load_london_msoa(GEO_PATH, df.rename(columns={'yee_modal': 'YeeModal'}))
fig, ax = plt.subplots(figsize=(11, 9))

plot_london_categorical(
    gdf, column='YeeModal',
    title='Yee & Dennett modal neighbourhood class (2001–2011)',
    color_dict=YEE_CLASS_COLORS, ax=ax)

# Create custom patches using your color dictionary
legend_handles = [
    mpatches.Patch(color=color, label=label) 
    for label, color in YEE_CLASS_COLORS.items()
]

# Add the legend to the axes
ax.legend(
    handles=legend_handles, 
    title='Class',         # Optional: Add a title to the legend
    loc='lower right',     # Adjust location as needed (e.g., 'upper left', 'best')
    frameon=True           # Set to False if you want no box around the legend
)
# ------------------------

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig02_map_yee_modal_class.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── export ─────────────────────────────────────────────────────────
out = OUT_DIR / f'yee_cascade_comparison_20260627.csv'
df.to_csv(out, index=False)
print(f'✓ saved {out}  ({df.shape[0]} x {df.shape[1]})')

--- 

## Flow Typology vs IMD change

In [ ]:
print("FLOW TYPOLOGY ONLY  vs  IMD_Pctile_Change")
for leg, typ, dom in [('2011','Typ_C_11','Dom_C_11'), ('2021','Typ_C_21','Dom_C_21')]:
    s = df.dropna(subset=[typ, 'IMD_Pctile_Change'])
    print(f"\n--- {leg} ({typ}) ---")
    print(s.groupby(typ)['IMD_Pctile_Change'].agg(['mean','median','count']).round(4).to_string())
    grps = [g['IMD_Pctile_Change'].dropna().values for _, g in s.groupby(typ) if len(g) > 1]
    H, p = stats.kruskal(*grps)
    k, n = len(grps), len(s)
    eps2 = (H - k + 1) / (n - k)                      # epsilon-squared effect size
    rho, pr = stats.spearmanr(s[dom], s['IMD_Pctile_Change'])
    print(f"  Kruskal-Wallis H={H:.1f} p={p:.1e} | epsilon^2={eps2:.3f}")
    print(f"  Spearman(Dom_C, IMD) rho={rho:+.3f} p={pr:.1e}")

### Interpretation:

***Cascade-flow detection is decoupled from the IMD-improvement that defines attribute-gentrification.***

- Yee's labels validate against deprivation, carrying the right sign for gentrification.
    - GEN label tracks IMD improvement: +0.042 mean percentile change.
    - DEC label tracks IMD decline: -0.0473 mean percentile change.
- **My flow typology does NOT track improvement the way GEN does.**
    - Cascade-led is flat: +0.009 in 2011 and -0.0058 in 2021.

In [ ]:
order = ['Cascade-led', 'Symmetric', 'Lateral', 'Counter-led']
ct = pd.crosstab(df['Typ_C_11'], df['Typ_C_21']).reindex(index=order, columns=order)
print("COUNTS:\n", ct.to_string())
print("\nROW % (where each 2011 class went):\n",
      (ct.div(ct.sum(axis=1), axis=0) * 100).round(1).to_string())

for ring in ['Inner', 'Outer']:
    s = df[df['Ring'] == ring]
    c = pd.crosstab(s['Typ_C_11'], s['Typ_C_21']).reindex(index=order, columns=order).fillna(0).astype(int)
    print(f"\n--- {ring} (n={len(s)}) ---\n", c.to_string())

### Interpretaion (transition matrix and ring split)

**Overall trend: Counter behaves like a one-way sink.**
- Counter-led is an absorbing state
    - 89% of 2011 counter areas stay counter.
- ***Cascade-led lost most***.
    - only 48% persist.
    - 26% flip to Counter
- The engine of the city-wide counter-shift is **Symmetric -> Counter** (126 MSOAs, 56%) and **Cascade -> Counter** (56 MSOAs).
- Only 15 change from Counter to Cascade.


***The ring split strongly support exodus from COVID.***
- ***Inner London cascade collapsed.***
    - 99 inner cascade-led 2011 MSOAs decreased to 18.
    - Among 99 in 2011, 
        - 43 flip to counter
        - 30 to symmetric
    - By 2021, only 29 Cascade-led MSOAs are from inner London.
- ***Outer London absorbed and generated cascade.***
    - Outer cascade is far stickier, with 85 of 114 (74.6%) persist.
    - 66 outer Symmetric MSOAs became cascade.
    - ~85% (166/195) of outer MSOAs supplies Cascade-led.

***The affluent-in-movement signal moved from inner to outer boroughs across the decade.***

### Limitation:

***2021 flow patterns are consistent with COVID-era suburbanisation, but we're not sure if caused by it.***
- We cannot separate a genuine structural shift from a transient pandemic artefact.